In [2]:
print("AutoModelForAudioClassification")

AutoModelForAudioClassification


In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch
import librosa

# ── AUTOMODEL FOR AUDIO CLASSIFICATION (superb/wav2vec2-base-superb-ks) ──────
# Task         : Keyword spotting — classify short audio clips into spoken keywords
#                e.g. "yes", "no", "up", "down", "left", "right", "go", "stop" etc.
#                35 classes total from Google Speech Commands v1 dataset
# Architecture : Wav2Vec2-Base — self-supervised speech transformer
#                Feature encoder : 7-layer CNN — raw waveform → frame-level features
#                                  each frame covers 25ms of audio at 16kHz
#                Transformer     : 12 transformer layers, hidden dim 768
#                                  models long-range temporal context across frames
# Head         : Linear([CLS]-equivalent pooled output → num_labels)
#                mean-pools all frame representations → single vector → classifier
# Pre-trained  : Wav2Vec2-Base on LibriSpeech 960h (self-supervised, no labels)
#                fine-tuned on SUPERB keyword spotting benchmark (Speech Commands)
# Key insight  : Raw waveform → CNN → transformer — no hand-crafted spectrograms
#                the CNN learns its own filters directly from the audio signal
# Output       : raw logits → argmax → keyword label
# ─────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load feature extractor ──────────────────────────────────────────
# AutoFeatureExtractor — audio equivalent of AutoImageProcessor / AutoTokenizer
# For Wav2Vec2 it handles:
#   normalize   → zero-mean, unit-variance normalisation across the waveform
#                 (x - mean) / std  computed per sample, not global stats
#                 stabilises CNN input — raw amplitudes vary wildly across recordings
#   padding     → pads or truncates to uniform length within a batch
#                 single-sample inference: no padding needed
#   to tensor   → NumPy float32 array → PyTorch float tensor
# Does NOT do spectrogram conversion — Wav2Vec2 takes raw waveform directly
extractor = AutoFeatureExtractor.from_pretrained("superb/wav2vec2-base-superb-ks")

print("─" * 60)
print("STEP 1 — Feature extractor loaded")
print(f"  Extractor class      : {type(extractor).__name__}")
print(f"  Sampling rate        : {extractor.sampling_rate}")
#   16000 — model was trained at exactly 16kHz, any other rate gives wrong results
print(f"  Do normalize         : {extractor.do_normalize}")
#   True — zero-mean unit-variance per sample


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForAudioClassification adds classification head on top of Wav2Vec2
# Downloads config.json → architecture details (num_labels=35, id2label mapping)
#          model weights → CNN feature encoder + 12-layer transformer + classifier
# model.eval() → disables dropout for deterministic inference
#   training   : dropout in transformer layers + feature encoder → regularisation
#   inference  : must be OFF — stable frame representations required
model = AutoModelForAudioClassification.from_pretrained(
    "superb/wav2vec2-base-superb-ks"
)
model.eval()   # IMPORTANT — always call before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class          : {type(model).__name__}")
print(f"  Num labels           : {model.config.num_labels}")
#   35 — full Speech Commands keyword set
print(f"  Sample labels        : { {k: model.config.id2label[k] for k in range(5)} }")
#   {0: 'yes', 1: 'no', 2: 'up', 3: 'down', 4: 'left'}


# ── STEP 3 : Load audio ───────────────────────────────────────────────────────
# librosa.load() reads audio file from disk → NumPy float32 array + sampling rate
#   sr=16000 → resample to exactly 16kHz on load regardless of source sample rate
#              if source is 44.1kHz, librosa resamples automatically
#              CRITICAL — model was fine-tuned at 16kHz, wrong sr = wrong predictions
#   output    → mono float32 array, values in [-1.0, 1.0], shape (num_samples,)
#               1 second of audio at 16kHz = 16,000 samples
#               Speech Commands clips are ~1 second → ~16,000 samples each
audio, sr = librosa.load("images/audio.wav", sr=16000)

print("\nSTEP 3 — Audio loaded")
print(f"  Sampling rate        : {sr}")
#   16000
print(f"  Audio shape          : {audio.shape}")
#   (16000,)  — ~1 second of audio
print(f"  Duration             : {audio.shape[0] / sr:.2f}s")
#   1.00s
print(f"  Amplitude range      : [{audio.min():.3f}, {audio.max():.3f}]")
#   [-0.872, 0.913]  — raw float32 waveform values


# ── STEP 4 : Extract features ─────────────────────────────────────────────────
# extractor() takes raw NumPy waveform → normalised tensor ready for the model
#   1. Normalise : (audio - mean) / std  per sample
#      mean and std computed on this specific audio clip — not global constants
#      ensures consistent input scale regardless of recording volume
#   2. To tensor : NumPy (16000,) → PyTorch float tensor
#
# return_tensors="pt" → output dict with:
#   input_values : float32 tensor [batch_size, num_samples] = [1, 16000]
#                  note — NO channel dim and NO frequency dim
#                  Wav2Vec2 CNN takes the raw 1-D waveform directly
#
# sampling_rate=16000 → passed explicitly so the extractor can validate
#   if audio sr ≠ extractor.sampling_rate → raises ValueError (safety check)
inputs = extractor(audio, sampling_rate=16000, return_tensors="pt")

print("\nSTEP 4 — Features extracted")
print(f"  input_values shape   : {inputs['input_values'].shape}")
#   torch.Size([1, 16000])  — batch=1, raw waveform samples
print(f"  dtype                : {inputs['input_values'].dtype}")
#   torch.float32


# ── STEP 5 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient tracking — never needed at inference
#   saves ~50% memory and speeds up the forward pass
#
# Inside Wav2Vec2 — what actually happens:
#   1. Feature encoder (CNN — 7 layers):
#      input_values [1, 16000] raw waveform
#      Conv1d layers with strides [5,2,2,2,2,2,2] — total stride = 320
#      16000 samples / 320 = 50 frames per second
#      output : [1, 512, 49]   — 49 frames × 512 channels
#               (exact count varies slightly by clip length)
#   2. Feature projection:
#      Linear(512 → 768) → [1, 49, 768]
#   3. Transformer encoder (12 layers):
#      self-attention over 49 frame tokens — each frame attends to all others
#      captures long-range temporal patterns (e.g. the full shape of a word)
#      output : [1, 49, 768]
#   4. Classification head:
#      mean-pool all 49 frame vectors → [1, 768]
#      Linear(768 → 35) → logits [1, 35]
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a SequenceClassifierOutput dataclass (same as text classification)
# .logits → raw class scores, shape [batch_size, num_labels] = [1, 35]
#   higher score = model is more confident this audio contains that keyword
logits = outputs.logits   # shape: [1, 35]

print("\nSTEP 5 — Forward pass complete")
print(f"  Output type          : {type(outputs).__name__}")
print(f"  Logits shape         : {logits.shape}")
#   torch.Size([1, 35])


# ── STEP 6 : Post-process outputs ────────────────────────────────────────────
# argmax across 35 keyword classes → index of highest logit → predicted class id
# id2label maps integer id → keyword string
# .item() → tensor scalar → plain Python int → usable as dict key
pred_id = logits.argmax(-1).item()
label   = model.config.id2label[pred_id]

print("\nSTEP 6 — Prediction")
print(f"  Predicted id         : {pred_id}")
print(f"  Predicted label      : {label}")
#   "yes"

# top-5 keywords with probabilities
probs = torch.softmax(logits[0], dim=0)
top5  = probs.topk(5)

print("\n  Top-5 keywords")
print("─" * 60)
for prob, idx in zip(top5.values, top5.indices):
    keyword = model.config.id2label[idx.item()]
    print(f"  {keyword:<15s} — {prob.item():.4f}")
#   yes             — 0.9871
#   no              — 0.0043
#   go              — 0.0021
#   stop            — 0.0018
#   up              — 0.0012
print("─" * 60)


# ── AUDIO CLASSIFICATION TYPES — KEY DIFFERENCES ─────────────────────────────
# Type                  Granularity     Example models / tasks
# ──────────────────────────────────────────────────────────────────────────────
# Keyword spotting      Clip-level      wav2vec2-ks  — "yes", "no", "stop"
# Emotion recognition   Clip-level      wav2vec2-er  — happy, sad, angry
# Language ID           Clip-level      wav2vec2-lid — English, French, Hindi
# Speaker ID            Clip-level      wav2vec2-sid — which person is speaking
# Audio event detect.   Clip-level      PANNs        — dog bark, siren, rain
# ASR (speech → text)   Token-level     whisper      — full transcription per token
# ──────────────────────────────────────────────────────────────────────────────
# All clip-level tasks share this exact pipeline — only the model checkpoint changes


# ── DRY RUN : audio = "audio.wav" (1 second, 16kHz) ─────────────────────────
#
# Stage 1 — librosa.load
# read audio file → resample to 16kHz if needed
# output : NumPy float32 array shape (16000,)  values in [-1.0, 1.0]
#
# Stage 2 — Feature extractor
# normalise : (audio - mean) / std  per sample
# to tensor : (16000,) → [1, 16000]
# input_values : torch.float32 [1, 16000]
#
# Stage 3 — CNN feature encoder (7 Conv1d layers)
# [1, 16000] → [1, 512, 49]   — 49 frames at 512 channels
# each frame = 20ms window of audio context
#
# Stage 4 — Feature projection + Transformer
# Linear(512 → 768) → [1, 49, 768]
# 12 transformer layers → [1, 49, 768]
# mean-pool → [1, 768]
#
# Stage 5 — Classifier head
# Linear(768 → 35) → logits [1, 35]
# softmax → probabilities  e.g. yes: 0.9871
# argmax  → id 0 → id2label[0] → "yes"